# Data Prep from Silver to Gold

Requires silver parquet data

In gold layer we add features (especial spatial features like hexagon) and aggregate data from different datasets, etc.

In [1]:
import pandas as pd
import duckdb
import polars as pl
import numpy as np
import geopandas as gpd
from shapely import wkt
import h3
from datetime import datetime
import math

BRONZE_CENSUS_TRACTS = "../data/processed_data/bronze_Census_Tracts.parquet"
BRONZE_COMMUNITY_AREA = "../data/processed_data/bronze_Community_Areas.parquet"

SILVER_TAXI_PATH = "../data/processed_data/silver_taxi.parquet"
SILVER_WEATHER_PATH = "../data/processed_data/silver_weatherdata.parquet"
SILVER_HEXAGON_PATH = "../data/processed_data/silver_dim_h3_chicago_8.parquet"

GOLD_TAXI_PATH = "../data/processed_data/gold_taxi.parquet"
GOLD_WEATHER_PATH = "../data/processed_data/gold_weather.parquet"
GOLD_HOURLY_DEMAND_HEXAGON = "../data/processed_data/GOLD_HOURLY_DEMAND_HEXAGON.parquet"
GOLD_HOURLY_DEMAND_CENSUS_TRACT = "../data/processed_data/GOLD_HOURLY_DEMAND_CENSUS_TRACT.parquet"
GOLD_HOURLY_DEMAND_COMMUNITY_AREA = "../data/processed_data/GOLD_HOURLY_DEMAND_COMMUNITY_AREA.parquet"

# Hexagon related 
H3_RESOLUTION = 8

# Weather related
START_TS = "2024-01-01 00:00:00"
END_TS = "2026-05-01 23:00:00" # also max date in taxi data

Nachdem in silver alle Duplikate bereinigt wurden, können nun trip_id und taxi_id gedropped werden

## Taxi Data

### Adding hexagon

First we add hexagon data and then compare the hexagons of trips against the city boundaries of chicago. We only want trips with pickup hexagon in the city boundaries

In [7]:
taxi_with_h3 = (
    pl.scan_parquet(SILVER_TAXI_PATH)
    .with_columns(
        pl.struct(["pickup_centroid_latitude", "pickup_centroid_longitude"])
        .map_elements(
            lambda row: h3.latlng_to_cell(
                row["pickup_centroid_latitude"],
                row["pickup_centroid_longitude"],
                H3_RESOLUTION,
            ),
            return_dtype=pl.String,
        )
        .alias("pickup_h3_cell"),
        
    )
)

# Check if there are trips without hexagon in Chicagos boundaries
valid_chicago_h3_cells = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select("h3_cell")
)

count_before = (
    taxi_with_h3
    .select(pl.len().alias("n_rows_before"))
    .collect()
)

taxi_with_h3_chicago_only = (
    taxi_with_h3
    .join(
        valid_chicago_h3_cells,
        left_on="pickup_h3_cell",
        right_on="h3_cell",
        how="inner",
    )
)

count_after = (
    taxi_with_h3_chicago_only
    .select(pl.len().alias("n_rows_after"))
    .collect()
)

print(count_before)
print(count_after)

shape: (1, 1)
┌───────────────┐
│ n_rows_before │
│ ---           │
│ u32           │
╞═══════════════╡
│ 13387587      │
└───────────────┘
shape: (1, 1)
┌──────────────┐
│ n_rows_after │
│ ---          │
│ u32          │
╞══════════════╡
│ 13387586     │
└──────────────┘


### Save gold version


In [8]:
taxi_with_h3_chicago_only.sink_parquet(GOLD_TAXI_PATH)

print(f"Silver taxi parquet written to: {GOLD_TAXI_PATH}")

Silver taxi parquet written to: ../data/processed_data/gold_taxi.parquet


## Weather Data

Create hourly indexed dataset for weather data from 01.01.2024 to 24.05.2026

In [9]:
def mode_or_na(series: pd.Series):
    """Return most frequent non-null value, otherwise NA."""
    mode_values = series.dropna().mode()
    if len(mode_values) == 0:
        return pd.NA
    return mode_values.iloc[0]


def create_hourly_weather_gold(
    df: pd.DataFrame,
    start_ts: str = START_TS,
    end_ts: str = END_TS,
) -> pd.DataFrame:
    """
    Create hourly gold weather dataset.

    Steps:
    - Parse valid timestamp
    - Restrict to requested date range
    - Aggregate observations to hourly level
    - Create complete hourly timestamp spine
    - Reindex to all hours
    - Linearly interpolate numeric weather columns
    - Fill categorical/context columns
    - Add date/hour helper columns
    """

    df = df.copy()

    # 1. Basic cleanup
    df["valid"] = pd.to_datetime(df["valid"], utc=True)
    df = df.dropna(subset=["valid"])
    
    # Convert UTC to chicago timezone
    df["valid"] = df["valid"].dt.tz_convert("America/Chicago")

    start_ts = (
        pd.Timestamp(start_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    end_ts = (
        pd.Timestamp(end_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    # Optional: keep only relevant date range
    df = df[(df["valid"] >= start_ts) & (df["valid"] <= end_ts)]

    # 2. Create hourly timestamp
    df["valid_hour"] = df["valid"].dt.floor("h", ambiguous=False, nonexistent="shift_forward")

    # 3. Cast numeric columns
    numeric_cols = [
        "tmpc",   # temperature Celsius
        "relh",   # relative humidity
        "sknt",   # wind speed in knots
        "p01m",   # precipitation
        "vsby",   # visibility
        "lat",
        "lon",
    ]

    existing_numeric_cols = [col for col in numeric_cols if col in df.columns]

    for col in existing_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Define aggregation rules
    agg_dict = {}

    # Numeric weather values: hourly mean
    for col in ["tmpc", "relh", "sknt", "vsby"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    # Precipitation: hourly sum is usually more sensible than mean
    if "p01m" in df.columns:
        agg_dict["p01m"] = "sum"

    # Station/location fields
    if "station" in df.columns:
        agg_dict["station"] = mode_or_na

    if "lat" in df.columns:
        agg_dict["lat"] = "mean"

    if "lon" in df.columns:
        agg_dict["lon"] = "mean"

    # Categorical weather condition
    if "skyc1" in df.columns:
        agg_dict["skyc1"] = mode_or_na

    # 5. Aggregate to hourly level
    hourly = (
        df
        .groupby("valid_hour", as_index=True)
        .agg(agg_dict)
        .sort_index()
    )

    # 6. Complete hourly index from start to end
    full_hourly_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq="h",
        name="valid_hour",
    )

    hourly = hourly.reindex(full_hourly_index)

    # 7. Interpolate numeric columns over missing hours
    numeric_interpolate_cols = [
        col for col in ["tmpc", "relh", "sknt", "p01m", "vsby", "lat", "lon"]
        if col in hourly.columns
    ]

    hourly[numeric_interpolate_cols] = (
        hourly[numeric_interpolate_cols]
        .interpolate(method="time", limit_direction="both")
    )

    # 8. Fill categorical/context columns
    categorical_fill_cols = [
        col for col in ["station", "skyc1"]
        if col in hourly.columns
    ]

    for col in categorical_fill_cols:
        hourly[col] = hourly[col].ffill().bfill()

    # 9. Back to normal dataframe
    gold = hourly.reset_index()

    # 10. Add helper columns for joins / ML
    # TODO might drop some later if not used for joining
    gold["date"] = gold["valid_hour"].dt.date
    gold["year"] = gold["valid_hour"].dt.year
    gold["month"] = gold["valid_hour"].dt.month
    gold["day"] = gold["valid_hour"].dt.day
    gold["hour"] = gold["valid_hour"].dt.hour
    gold["weekday"] = gold["valid_hour"].dt.weekday

    # 11. Optional quality flags
    original_hours = set(df["valid_hour"].dropna().unique())

    gold["was_observed_hour"] = gold["valid_hour"].isin(original_hours)
    gold["was_interpolated_hour"] = ~gold["was_observed_hour"]
    
    # Onehot encoding
    skyc1_dummies = pd.get_dummies(
        gold["skyc1"],
        prefix="skyc1",
        dummy_na=False,
        dtype="int8",
    )

    gold = pd.concat([gold, skyc1_dummies], axis=1)
    
    # Drop unused columns
    gold = gold.drop(columns=["skyc1", "station", "lat", "lon"])

    return gold

weather_silver = pd.read_parquet(SILVER_WEATHER_PATH)

weather_gold = create_hourly_weather_gold(
    weather_silver,
    start_ts=START_TS,
    end_ts=END_TS,
)

weather_gold.to_parquet(
    GOLD_WEATHER_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print(f"Gold weather parquet written to: {GOLD_WEATHER_PATH}")
print(f"Rows: {len(weather_gold):,}")
print(f"Start: {weather_gold['valid_hour'].min()}")
print(f"End: {weather_gold['valid_hour'].max()}")
print()
print(weather_gold.head().to_string(index=False))
print()
print(weather_gold.tail().to_string(index=False))

Gold weather parquet written to: ../data/processed_data/gold_weather.parquet
Rows: 20,447
Start: 2024-01-01 00:00:00-06:00
End: 2026-05-01 23:00:00-05:00

               valid_hour  tmpc  relh  sknt  vsby  p01m       date  year  month  day  hour  weekday  was_observed_hour  was_interpolated_hour  skyc1_BKN  skyc1_CLR  skyc1_FEW  skyc1_OVC  skyc1_SCT  skyc1_VV 
2024-01-01 00:00:00-06:00  1.11 75.26  12.0  10.0   0.0 2024-01-01  2024      1    1     0        0               True                  False          0          0          0          1          0          0
2024-01-01 01:00:00-06:00  1.11 75.26  12.0   8.0   0.0 2024-01-01  2024      1    1     1        0               True                  False          0          0          0          1          0          0
2024-01-01 02:00:00-06:00  0.56 81.63  10.0   7.0   0.0 2024-01-01  2024      1    1     2        0               True                  False          0          0          0          1          0          0
2024-01-01 03

In [10]:
duckdb.sql(f"""
    SELECT hour, count(*)
    FROM  read_parquet('{GOLD_WEATHER_PATH}')
    GROUP BY hour
    ORDER BY hour asc       
           """).show(max_rows=100)

duckdb.sql(f"""
    SELECT date, hour, was_interpolated_hour
    FROM  read_parquet('{GOLD_WEATHER_PATH}')
    WHERE was_interpolated_hour = True    
           """).show(max_rows=100)

┌───────┬──────────────┐
│ hour  │ count_star() │
│ int32 │    int64     │
├───────┼──────────────┤
│     0 │          852 │
│     1 │          854 │
│     2 │          849 │
│     3 │          852 │
│     4 │          852 │
│     5 │          852 │
│     6 │          852 │
│     7 │          852 │
│     8 │          852 │
│     9 │          852 │
│    10 │          852 │
│    11 │          852 │
│    12 │          852 │
│    13 │          852 │
│    14 │          852 │
│    15 │          852 │
│    16 │          852 │
│    17 │          852 │
│    18 │          852 │
│    19 │          852 │
│    20 │          852 │
│    21 │          852 │
│    22 │          852 │
│    23 │          852 │
└───────┴──────────────┘
  24 rows    2 columns

┌────────────┬───────┬───────────────────────┐
│    date    │ hour  │ was_interpolated_hour │
│    date    │ int32 │        boolean        │
├────────────┼───────┼───────────────────────┤
│ 2024-11-03 │     1 │ true                  │
│ 2025-03-09 │  

In [11]:
duckdb.sql("SET TimeZone='America/Chicago'")
duckdb.sql(f"""
    SELECT date, count(*)
    FROM  read_parquet('{GOLD_WEATHER_PATH}')
    GROUP BY date
    HAVING count(*) <> 24
    ORDER BY date    
           """).show(max_rows=100)

duckdb.sql(f"""
    SELECT count(*)
    FROM  read_parquet('{GOLD_WEATHER_PATH}')  
           """).show(max_rows=100)

┌────────────┬──────────────┐
│    date    │ count_star() │
│    date    │    int64     │
├────────────┼──────────────┤
│ 2024-03-10 │           23 │
│ 2024-11-03 │           25 │
│ 2025-03-09 │           23 │
│ 2025-11-02 │           25 │
│ 2026-03-08 │           23 │
└────────────┴──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        20447 │
└──────────────┘



## Hourly Dataset

### Preparations

Here we create an hourly dataset, whereby each hour contains a row for each hexagon. 

We use cyclic encoding for columns like hour, weekday or month to be aware of the distance between time instances (e. g. hour 23 -> 0).

And we merge with the weather data.

These steps are done before the cross join as they are only dependend on time and not on the spatial unit

In [2]:
CHICAGO_TZ = "America/Chicago"


# --------------------------------------------------
# 1. Lokale Stunden erzeugen, aber timezone-naive
# --------------------------------------------------

hourly_index = pd.date_range(
    start=START_TS,
    end=END_TS,
    freq="h",
    inclusive="both",
    name="datetime_hour",
)

hours_pd = pd.DataFrame({
    "datetime_hour": hourly_index
})

hours = (
    pl.from_pandas(hours_pd)
    .with_columns(
        pl.col("datetime_hour")
        .cast(pl.Datetime("us"))
        .dt.truncate("1h")
        .alias("datetime_hour")
    )
)


# --------------------------------------------------
# 2. Zeitfeatures vor dem Cross Join hinzufügen
# --------------------------------------------------

hours_with_features = (
    hours
    .with_columns([
        pl.col("datetime_hour").dt.month().alias("month"),
        pl.col("datetime_hour").dt.weekday().alias("weekday"),
        pl.col("datetime_hour").dt.hour().alias("hour"),
    ])
    .with_columns([
        ((2 * math.pi * (pl.col("month") - 1) / 12).sin()).alias("month_sin"),
        ((2 * math.pi * (pl.col("month") - 1) / 12).cos()).alias("month_cos"),

        ((2 * math.pi * (pl.col("weekday") - 1) / 7).sin()).alias("weekday_sin"),
        ((2 * math.pi * (pl.col("weekday") - 1) / 7).cos()).alias("weekday_cos"),

        ((2 * math.pi * pl.col("hour") / 24).sin()).alias("hour_sin"),
        ((2 * math.pi * pl.col("hour") / 24).cos()).alias("hour_cos"),
    ])
)


# --------------------------------------------------
# 3. Wetterdaten auf denselben Join-Key bringen
# --------------------------------------------------

weather_hourly = (
    pl.scan_parquet(GOLD_WEATHER_PATH)
    .with_columns(
        pl.col("valid_hour")
        .dt.convert_time_zone(CHICAGO_TZ)
        .dt.truncate("1h")
        .dt.replace_time_zone(None)   # wichtig: timezone-naive machen
        .cast(pl.Datetime("us"))      # exakt gleicher Typ wie hours
        .alias("datetime_hour")
    )
    .group_by("datetime_hour")
    .agg(
        pl.all()
        .exclude(["valid_hour", "datetime_hour"])
        .first()
    )
)


# --------------------------------------------------
# 4. Weather vor dem Cross Join an Hours joinen
# --------------------------------------------------

hours_with_weather = (
    hours_with_features
    .lazy()
    .join(
        weather_hourly,
        on="datetime_hour",
        how="left",
    )
).lazy()


# --------------------------------------------------
# 5. Testweise speichern
# --------------------------------------------------

test_path = "../data/processed_data/test_daterange.parquet"

hours_with_weather.sink_parquet(test_path)

In [3]:
duckdb.sql(f"""
    SELECT count(*) AS n_rows
    FROM read_parquet('{test_path}')
""").show()

┌────────┐
│ n_rows │
│ int64  │
├────────┤
│  20448 │
└────────┘



### Hexagon Hourly

In [4]:
hexagon = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select([
        pl.col("h3_cell").alias("h3_cell"),  
    ])
    .unique()
    .collect()
).lazy()

hexagon_hourly = (
    hours_with_weather
    .join(
        hexagon,
        how="cross",
    )
)

hexagon_hourly.sink_parquet(GOLD_HOURLY_DEMAND_HEXAGON)

/var/folders/sw/7cvjbzt5623085bdv7hpg6vw0000gn/T/ipykernel_5725/3666555770.py:7: UserWarning: Extension type 'geoarrow.wkb' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  .collect()


In [5]:
duckdb.sql(f"""
    SELECT count(*)
    FROM  read_parquet('{GOLD_HOURLY_DEMAND_HEXAGON}')      
           """).show(max_rows=100)

duckdb.sql(f"""
    SELECT EXTRACT('hour' FROM datetime_hour), count(*)
    FROM  read_parquet('{GOLD_HOURLY_DEMAND_HEXAGON}')
    GROUP BY EXTRACT('hour' FROM datetime_hour)
    ORDER BY EXTRACT('hour' FROM datetime_hour) asc       
           """).show(max_rows=100)



┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     17442144 │
└──────────────┘

┌───────────────────────────────────────┬──────────────┐
│ main.date_part('hour', datetime_hour) │ count_star() │
│                 int64                 │    int64     │
├───────────────────────────────────────┼──────────────┤
│                                     0 │       726756 │
│                                     1 │       726756 │
│                                     2 │       726756 │
│                                     3 │       726756 │
│                                     4 │       726756 │
│                                     5 │       726756 │
│                                     6 │       726756 │
│                                     7 │       726756 │
│                                     8 │       726756 │
│                                     9 │       726756 │
│                                    10 │       726756 │
│                                    11 │ 

### Census Tract Hourly

In [6]:
census_tracts = (
    pl.scan_parquet(BRONZE_CENSUS_TRACTS)
    .select([
        pl.col("CENSUS_T_1").alias("census_tract"),  
    ])
    .unique()
    .collect()
).lazy()

census_tract_hourly = (
    hours_with_weather
    .join(
        census_tracts,
        how="cross",
    )
)

census_tract_hourly.sink_parquet(GOLD_HOURLY_DEMAND_CENSUS_TRACT)

In [7]:
n_hours = (
    hours_with_weather
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
n_census_tracts = (
    census_tracts
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
actual_rows = duckdb.sql(f"""
SELECT
    COUNT(*) AS n_rows,
FROM read_parquet('{GOLD_HOURLY_DEMAND_CENSUS_TRACT}')
""").df()["n_rows"].iloc[0]

print(n_hours * n_census_tracts)
print(actual_rows)

17953344
17953344


### Community Area Hourly

In [8]:
community_area = (
    pl.scan_parquet(BRONZE_COMMUNITY_AREA)
    .select(
        pl.col("AREA_NUM_1").alias("community_area")
    )
    .unique()
    .collect()
).lazy()

community_area_hourly = (
    hours_with_weather
    .join(
        community_area,
        how="cross",
    )
)

community_area_hourly.sink_parquet(GOLD_HOURLY_DEMAND_COMMUNITY_AREA)

In [9]:
n_hours = (
    hours_with_weather
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
n_community_areas = (
    community_area
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
actual_rows = duckdb.sql(f"""
SELECT
    COUNT(*) AS n_rows,
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
""").df()["n_rows"].iloc[0]

print(n_hours * n_community_areas)
print(actual_rows)

1574496
1574496


In [10]:
duckdb.sql(f"""
SELECT
    *,
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
""").show()

┌─────────────────────┬───────┬─────────┬──────┬─────────────────────┬────────────────────┬─────────────────────┬──────────────────────┬──────────────────────┬───────────────────────┬────────┬────────┬───────────────────┬────────────────────┬─────────────────────┬────────────┬───────┬─────────────┬───────┬────────────┬───────────────┬───────────────────┬───────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┬───────────┬────────────────┐
│    datetime_hour    │ month │ weekday │ hour │      month_sin      │     month_cos      │     weekday_sin     │     weekday_cos      │       hour_sin       │       hour_cos        │  tmpc  │  relh  │       sknt        │        vsby        │        p01m         │    date    │ year  │ month_right │  day  │ hour_right │ weekday_right │ was_observed_hour │ was_interpolated_hour │ skyc1_BKN │ skyc1_CLR │ skyc1_FEW │ skyc1_OVC │ skyc1_SCT │ skyc1_VV  │ community_area │
│      timestamp      │ int8  │  int8   │ int8 │       doubl

In [ ]:
# TODO Daten aus Taxi rein packen in hourly_demand
# TODO visualisierung von verschiedenen Spatial Einheiten: Hexagon, Census Track, Community Area -> überlegen wie damit umzugehen ist
# TODO Openstreetmap data für POI

# TODO Target Target -> Demand: trips_started


### Function to aggregate taxi data into the spatio-temporal datasets

In [11]:
def add_taxi_data(
    skeleton: pl.LazyFrame | pl.DataFrame,
    taxi: pl.LazyFrame | pl.DataFrame,
    spatial_col: str,
    taxi_spatial_col: str,
    timestamp_col: str = "trip_start_timestamp",
    datetime_col: str = "datetime_hour",
) -> pl.LazyFrame:
    """
    Adds hourly trip-start demand to a complete hour × spatial-unit skeleton.

    Parameters
    ----------
    skeleton:
        Complete base dataset with one row per datetime_col × spatial_col.
        Example: datetime_hour × h3_cell, datetime_hour × census_tract, etc.

    taxi:
        Taxi trip dataset with one row per trip.

    spatial_col:
        Spatial column name in the skeleton/output.
        Example: "h3_cell", "census_tract", "community_area".

    taxi_spatial_col:
        Spatial pickup column name in taxi.
        Example: "pickup_h3_cell", "pickup_census_tract", "pickup_community_area".

    timestamp_col:
        Taxi trip start timestamp column.

    datetime_col:
        Hourly timestamp column used in skeleton/output.

    demand_col:
        Output demand column name.

    Returns
    -------
    pl.LazyFrame
        Skeleton enriched with demand_col.
    """

    if isinstance(skeleton, pl.DataFrame):
        skeleton = skeleton.lazy()

    if isinstance(taxi, pl.DataFrame):
        taxi = taxi.lazy()

    taxi_demand = (
        taxi
        .filter(
            pl.col(timestamp_col).is_not_null()
            & pl.col(taxi_spatial_col).is_not_null() # TODO Census Tract contains nulls
        )
        .with_columns([
            pl.col(timestamp_col)
            .cast(pl.Datetime("us"))
            .dt.truncate("1h")
            .alias(datetime_col),

            pl.col(taxi_spatial_col).alias(spatial_col),
        ])
        .group_by([
            datetime_col,
            spatial_col,
        ])
        .agg(
            # aggregations:
            pl.len().alias('trip_count'),
            pl.col('trip_seconds').sum().alias('trip_seconds_sum'),
            pl.col('trip_seconds').mean().alias('trip_seconds_mean'),
            pl.col('trip_seconds').min().alias('trip_seconds_min'),
            pl.col('trip_seconds').max().alias('trip_seconds_max'),
            pl.col('trip_miles').sum().alias('trip_miles_sum'),
            pl.col('trip_miles').mean().alias('trip_miles_mean'),
            pl.col('trip_miles').min().alias('trip_miles_min'),
            pl.col('trip_miles').max().alias('trip_miles_max'),
            pl.col('fare').sum().alias('fare_sum'),
            pl.col('fare').mean().alias('fare_mean'),
            pl.col('fare').min().alias('fare_min'),
            pl.col('fare').max().alias('fare_max'),
            pl.col('tips').sum().alias('tips_sum'),
            pl.col('tips').mean().alias('tips_mean'),
            pl.col('tips').min().alias('tips_min'),
            pl.col('tips').max().alias('tips_max'),
            pl.col('tolls').sum().alias('tolls_sum'),
            pl.col('tolls').mean().alias('tolls_mean'),
            pl.col('tolls').min().alias('tolls_min'),
            pl.col('tolls').max().alias('tolls_max'),
            pl.col('extras').sum().alias('extras_sum'),
            pl.col('extras').mean().alias('extras_mean'),
            pl.col('extras').min().alias('extras_min'),
            pl.col('extras').max().alias('extras_max'),
            pl.col('trip_total').sum().alias('trip_total_sum'),
            pl.col('trip_total').mean().alias('trip_total_mean'),
            pl.col('trip_total').min().alias('trip_total_min'),
            pl.col('trip_total').max().alias('trip_total_max'),
            pl.col('payment_type').drop_nulls().mode().first().alias('most_common_payment_type')
        )    
    )

    result = (
        skeleton
        .with_columns(
            pl.col(datetime_col)
            .cast(pl.Datetime("us"))
            .dt.truncate("1h")
            .alias(datetime_col)
        )
        .join(
            taxi_demand,
            on=[datetime_col, spatial_col],
            how="left",
        )
    )

    return result

In [12]:
hexagon_hourly_demand = add_taxi_data(
    skeleton=hexagon_hourly,
    taxi=pl.scan_parquet(GOLD_TAXI_PATH),
    spatial_col="h3_cell",
    taxi_spatial_col="pickup_h3_cell",
)

hexagon_hourly_demand.sink_parquet(GOLD_HOURLY_DEMAND_HEXAGON)

community_area_hourly_demand = add_taxi_data(
    skeleton=community_area_hourly,
    taxi=pl.scan_parquet(GOLD_TAXI_PATH),
    spatial_col="community_area",
    taxi_spatial_col="pickup_community_area",
)

community_area_hourly_demand.sink_parquet(GOLD_HOURLY_DEMAND_COMMUNITY_AREA)

census_tract_hourly_demand = add_taxi_data(
    skeleton=census_tract_hourly,
    taxi=pl.scan_parquet(GOLD_TAXI_PATH),
    spatial_col="census_tract",
    taxi_spatial_col="pickup_census_tract",
)

census_tract_hourly_demand.sink_parquet(GOLD_HOURLY_DEMAND_CENSUS_TRACT)

In [14]:
duckdb.sql(f"""
SELECT
    *
FROM read_parquet('{GOLD_HOURLY_DEMAND_HEXAGON}')
WHERE trip_count <> 0
LIMIT 10
""").show()

┌─────────────────────┬───────┬─────────┬──────┬────────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬─────────────────────┬──────────────────────┬────────┬────────┬────────┬────────┬────────┬────────────┬───────┬─────────────┬───────┬────────────┬───────────────┬───────────────────┬───────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┬───────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬──────────────────┬──────────────────┬────────────────────┬───────────────────┬────────────────┬────────────────┬──────────┬───────────┬──────────┬──────────┬──────────┬───────────┬──────────┬──────────┬───────────┬────────────┬───────────┬───────────┬────────────┬─────────────┬────────────┬────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬──────────────────────────┐
│    datetime_hour    │ month │ weekday │ hour │       month_sin        │        month_cos        │     we

In [16]:
duckdb.sql(f"""
SELECT
    *
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
WHERE trip_count <> 0
LIMIT 10
""").show()

┌─────────────────────┬───────┬─────────┬──────┬────────────────────┬───────────────────────┬─────────────────────┬──────────────────────┬─────────────────────┬──────────────────────┬───────────────────┬───────────────────┬───────────────────┬────────┬────────────────────┬────────────┬───────┬─────────────┬───────┬────────────┬───────────────┬───────────────────┬───────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┬───────────┬────────────────┬────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬────────────────────┬───────────────────┬────────────────┬────────────────┬──────────┬────────────────────┬──────────┬──────────┬──────────┬───────────┬──────────┬──────────┬───────────┬────────────┬───────────┬───────────┬────────────┬─────────────┬────────────┬────────────┬────────────────┬────────────────────┬────────────────┬────────────────┬──────────────────────────┐
│    datetime_hour    │ month │ weekday │ hour │     mo

In [17]:
duckdb.sql(f"""
SELECT
    *
FROM read_parquet('{GOLD_HOURLY_DEMAND_CENSUS_TRACT}')
WHERE trip_count <> 0
LIMIT 10
""").show()

┌─────────────────────┬───────┬─────────┬──────┬─────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬────────────────────────┬─────────────────────────┬────────────────────┬───────────────────┬────────┬────────┬────────┬────────────┬───────┬─────────────┬───────┬────────────┬───────────────┬───────────────────┬───────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┬───────────┬──────────────┬────────────┬──────────────────┬───────────────────┬──────────────────┬──────────────────┬────────────────────┬───────────────────┬────────────────┬────────────────┬──────────┬───────────┬──────────┬──────────┬──────────┬───────────┬──────────┬──────────┬───────────┬────────────┬───────────┬───────────┬────────────┬─────────────┬────────────┬────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬──────────────────────────┐
│    datetime_hour    │ month │ weekday │ hour │      month_sin      │        month